In [ ]:
import sys
import os

# This tells Python to look one folder up from 'mynotebooks' (back to your project root)
sys.path.append(os.path.abspath('..'))

import importlib
import helpers.openaiclient_helper
import helpers.rag_helper
# Force Jupyter to read the freshly saved files from your hard drive
importlib.reload(helpers.openaiclient_helper)
importlib.reload(helpers.rag_helper)

from helpers.openaiclient_helper import ClientHelper
from helpers.rag_helper import RAGBase
from helpers.ingest import load_faq_data, build_index
from gitsource import chunk_documents

selected_model="openai/gpt-oss-120b"
documents = load_faq_data()

chunks = chunk_documents(documents, size=2000, step=1000)
index = build_index(chunks)

In [ ]:
client = ClientHelper()
openai_client = client.get_client()
assistant = RAGBase(
    index=index,
    llm_client=openai_client
)

In [ ]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [ ]:
messages = [
    {'role': 'user', 'content': 'How does the agentic loop work, and how is it different from plain RAG?'}
]


In [ ]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [ ]:


response = openai_client.responses.create(
    model=selected_model,
    input=messages,
    tools=[search_tool]
)

len(response.output)
call = response.output[0]

In [ ]:
import json

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = assistant.search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [ ]:

def agent_loop(instructions, question, model=selected_model) -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False
        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [ ]:
question = 'How does the agentic loop work, and how is it different from plain RAG?'
result = agent_loop(instructions, question)

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(assistant.search, search_tool)

In [ ]:
agent_tools.get_tools()

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [ ]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(client=openai_client,  model=selected_model)
)

In [ ]:
result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback,
)

In [ ]:
runner.run();